# 01 - Agency Performance Overview

**Business question:** How is the Agency business performing overall?

This notebook establishes the portfolio baseline before deeper persistency and lapse-risk analysis.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "raw"

agents = pd.read_csv(DATA / "agents.csv", parse_dates=["join_date", "exit_date"])
policies = pd.read_csv(
    DATA / "policies.csv",
    parse_dates=["issue_date", "maturity_date", "lapse_date", "surrender_date"],
)
leads = pd.read_csv(DATA / "leads.csv", parse_dates=["lead_date", "conversion_date"])
targets = pd.read_csv(DATA / "sales_targets.csv", parse_dates=["target_month"])
payments = pd.read_csv(DATA / "premium_payments.csv", parse_dates=["due_date", "payment_date"])

agents.shape, policies.shape, payments.shape

## Core portfolio KPIs

In [ ]:
as_of = pd.Timestamp("2026-09-01")
current_month = as_of.to_period("M")

monthly_policies = policies[policies["issue_date"].dt.to_period("M") == current_month]
active_agents = agents[agents["status"] == "Active"]

kpis = {
    "New policies": len(monthly_policies),
    "Annualized premium": monthly_policies["annualized_premium"].sum(),
    "Registered active agents": len(active_agents),
    "Policies per active agent": len(monthly_policies) / max(len(active_agents), 1),
    "Current active policies": (policies["policy_status"] == "Active").sum(),
    "Current lapsed policies": (policies["policy_status"] == "Lapsed").sum(),
}
pd.Series(kpis)

## Agency-unit performance

In [ ]:
agency_units = pd.read_csv(DATA / "agency_units.csv")
agent_unit = agents[["agent_id", "agency_unit_id"]]

monthly = monthly_policies.merge(agent_unit, on="agent_id").merge(
    agency_units[["agency_unit_id", "agency_name", "region"]], on="agency_unit_id"
)

unit_perf = (
    monthly.groupby(["agency_unit_id", "agency_name", "region"], as_index=False)
    .agg(new_policies=("policy_id", "count"), annualized_premium=("annualized_premium", "sum"))
    .sort_values("annualized_premium", ascending=False)
)
unit_perf.head(12)

## Next analysis questions
- Which agents are driving the result?
- Which units are below target?
- Is higher sales volume associated with weaker persistency?
- Which payment behaviours precede lapse?